# CompVP Project Part 1 - Mooney faces vs. computer vision systems
1. Select 100 strongly side-lit faces from LFW (one per identity)
2. Generate Mooney images: blur + threshold fixed by percentage of white pixels (Egger et al.)
3. Vision system 1: Depth Anything V2 Small (monocular relative depth)
4. Vision system 2: CLIP ViT-B/32 (zero-shot recognition + 8AFC identity matching as in the paper)

In [ ]:
N_IMAGES = 100
SIZE = 256          # output resolution of originals and Mooneys
CROP = 0.62         # central fraction of the 250x250 LFW frame that is kept (removes most background)
WHITE_PCT = 30.0    # percentage of white pixels
BLUR_REL = 0.04     # blur kernel size relative to image side (paper: 7px @ 250px = 0.028)
SIGMA = 35.0        # Gaussian sigma (paper value)
MIN_FACE_WHITE = 2 / 3  # min share of the Mooney's white pixels inside the central face region (rejects bright backgrounds)
SUBSET = [3, 11, 22, 27, 35, 45, 54, 63, 77, 84]  # top 10, picked from figures/sheet_mooney.jpg of run v4
SEED = 0
DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"
CLIP_ID = "openai/clip-vit-base-patch32"

In [ ]:
%%writefile mooney_tool.py
"""Mooney image generator (CLI, batch capable).

Gaussian blur followed by a global threshold that is found by binary search so that a
fixed percentage of pixels ends up white (Egger et al., "Inverse Rendering Best Explains
Face Perception Under Extreme Illuminations").

Examples:
    python mooney_tool.py photo.jpg out/ --white 16
    python mooney_tool.py Original/ Mooney/ --white 35 --blur-rel 0.04 --log params.csv
"""
import argparse
import csv
from pathlib import Path

import cv2
import numpy as np

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def kernel_size(h: int, w: int, blur_rel: float) -> int:
    k = int(round(blur_rel * min(h, w)))
    return max(1, k + (1 if k % 2 == 0 else 0))


def white_threshold(img: np.ndarray, white_frac: float, iters: int = 50) -> float:
    lo, hi = float(img.min()) - 1e-3, float(img.max())
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if (img > mid).mean() > white_frac:
            lo = mid
        else:
            hi = mid
    return hi


def fill_ties(mask: np.ndarray, blurred: np.ndarray, gray: np.ndarray, target: int, k: int) -> int:
    """Pixels with identical blurred values can make the exact white count unreachable by any threshold.
    Fill the shortfall from the tied pixels at the boundary, ranked by brightness in a wider blur."""
    deficit = target - int(mask.sum())
    if deficit <= 0:
        return 0
    tied = np.flatnonzero((blurred == blurred[~mask].max()) & ~mask)
    wide = cv2.blur(gray, (3 * k, 3 * k)).ravel()[tied]
    mask.flat[tied[np.argsort(-wide, kind="stable")[:deficit]]] = True
    return deficit


def make_mooney(image: np.ndarray, white_pct: float = 16.0, blur_rel: float = 0.028,
                sigma: float = 35.0, ksize: int = 0, invert: bool = False):
    """Return (mooney uint8 image, parameter dict). `image` is BGR or grayscale."""
    gray = image if image.ndim == 2 else cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32)
    k = ksize if ksize > 0 else kernel_size(gray.shape[0], gray.shape[1], blur_rel)
    if k % 2 == 0:
        k += 1
    # With sigma >> ksize (paper: 7x7, sigma=35) this is close to a box blur of size k.
    blurred = cv2.GaussianBlur(gray, (k, k), sigma) if k > 1 else gray
    t = white_threshold(blurred, white_pct / 100.0)
    white = blurred > t
    tie_filled = fill_ties(white, blurred, gray, int(np.floor(white.size * white_pct / 100.0 + 1e-6)), k)
    mooney = np.where(white, 255, 0).astype(np.uint8)
    if invert:
        mooney = 255 - mooney
    info = {
        "ksize": k,
        "sigma": sigma,
        "threshold": round(t, 3),
        "white_pct_target": white_pct,
        "white_pct_actual": round(100.0 * float((mooney > 0).mean()), 2),
        "tie_filled_px": tie_filled,
        "inverted": invert,
    }
    return mooney, info


def iter_images(path: Path):
    if path.is_file():
        return [path]
    return sorted(p for p in path.iterdir() if p.suffix.lower() in IMAGE_EXTS)


def main(argv=None):
    ap = argparse.ArgumentParser(description="Generate Mooney images (blur + white-pixel-percentage threshold).")
    ap.add_argument("input", type=Path, help="image file or folder of images")
    ap.add_argument("output", type=Path, help="output folder (PNG files, same stem as input)")
    ap.add_argument("--white", type=float, default=16.0,
                    help="percentage of white pixels, 0-100 (default 16, as in Egger et al.)")
    ap.add_argument("--blur-rel", type=float, default=0.028,
                    help="blur strength: kernel size as fraction of the shorter image side (7px @ 250px = 0.028)")
    ap.add_argument("--ksize", type=int, default=0, help="absolute kernel size in px (overrides --blur-rel)")
    ap.add_argument("--sigma", type=float, default=35.0, help="Gaussian sigma (default 35, as in the paper)")
    ap.add_argument("--invert", action="store_true", help="flip contrast polarity")
    ap.add_argument("--log", type=Path, help="optional CSV file recording the parameters per image")
    a = ap.parse_args(argv)

    if not 0 < a.white < 100:
        ap.error("--white must be between 0 and 100")
    if not a.input.exists():
        ap.error(f"input not found: {a.input}")
    files = iter_images(a.input)
    if not files:
        ap.error(f"no images found in {a.input}")
    a.output.mkdir(parents=True, exist_ok=True)

    rows = []
    for i, f in enumerate(files, 1):
        img = cv2.imread(str(f), cv2.IMREAD_COLOR)
        if img is None:
            print(f"[{i}/{len(files)}] skipped unreadable file {f.name}")
            continue
        mooney, info = make_mooney(img, a.white, a.blur_rel, a.sigma, a.ksize, a.invert)
        cv2.imwrite(str(a.output / f"{f.stem}.png"), mooney)
        rows.append({"file": f.name, **info})
        print(f"[{i}/{len(files)}] {f.name}: k={info['ksize']} white={info['white_pct_actual']:.1f}%")

    if a.log and rows:
        a.log.parent.mkdir(parents=True, exist_ok=True)
        with open(a.log, "w", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    print(f"Done: {len(rows)} Mooney images written to {a.output}")


if __name__ == "__main__":
    main()

In [ ]:
import os, sys, time, faulthandler
os.environ["HF_HUB_OFFLINE"] = "1"  # must be set before transformers / huggingface_hub are imported
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import pearsonr, spearmanr, binomtest
from mooney_tool import make_mooney

T0 = time.time()
def watchdog(minutes, stage):
    """If `stage` takes longer than `minutes`, dump every thread's stack to the Kaggle log and stop the kernel."""
    faulthandler.dump_traceback_later(minutes * 60, exit=True, file=sys.__stderr__)
    print(f"[{time.time() - T0:7.1f}s] {stage} (watchdog {minutes} min)", flush=True)

watchdog(30, "setup, model check, image preparation")

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("work").resolve()
EXPORT = WORK / "export"
OUT, ANA, FIG = EXPORT / "deliverable", EXPORT / "analysis", EXPORT / "figures"
shutil.rmtree(EXPORT, ignore_errors=True)
for d in (OUT / "Original", OUT / "Mooney", ANA, FIG):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__, "| CPU threads", torch.get_num_threads(), "/", os.cpu_count())

## Offline model setup (no internet, GPU optional)
Both vision models are attached as Kaggle inputs and loaded with `local_files_only=True`.
The Kaggle copies are third-party uploads, so every required file is checked against the official
Hugging Face release (size + git blob SHA-1, or LFS SHA-256 for weights) before anything is loaded.

In [ ]:
import hashlib
import transformers
from transformers import AutoImageProcessor, AutoModelForDepthEstimation, CLIPModel, CLIPProcessor

MODEL_SPECS = {
    "depth": {
        "name": "Depth Anything V2 Small",
        "official": f"huggingface.co/{DEPTH_ID} @ 5426e4f0f36572d16453bbda7a8389317b1bef99",
        "attach": "Kaggle model 'itsraininnnn/depth-anything-v2-small-hf' (Transformers, variation 'default', version 1)",
        "model_type": "depth_anything",
        "files": {  # name: (bytes, hash type, official hash)
            "config.json": (950, "git", "9207d3a304e86517dab21638c559d8b3195d41b6"),
            "preprocessor_config.json": (775, "git", "a7822a90a0e91a14d50d1996953bac8f389a4377"),
            "model.safetensors": (99173660, "sha256", "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"),
        },
    },
    "clip": {
        "name": "CLIP ViT-B/32",
        "official": f"huggingface.co/{CLIP_ID} @ 3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268",
        "attach": "Kaggle dataset 'singhaltarun/openaiclip-vit-base-patch32'",
        "model_type": "clip",
        "files": {
            "config.json": (4186, "git", "a2a88b96561196777ca173b15309ea859f4d2ce0"),
            "preprocessor_config.json": (316, "git", "5a12a1eb250987a4eee0e3e7d7338c4b22724be1"),
            "tokenizer_config.json": (592, "git", "4fdaf6842dd5a725b940c92b6b692490ce59d548"),
            "vocab.json": (862328, "git", "182766ce89b439768edadda342519f33802f5364"),
            "merges.txt": (524657, "git", "bbfec752c9a675946c6dce106def6f35c882dcc2"),
            "pytorch_model.bin": (605247071, "sha256", "a63082132ba4f97a80bea76823f544493bffa8082296d62d71581a4feff1576f"),
        },
    },
}

def file_hash(path, kind):
    h = hashlib.sha1(f"blob {path.stat().st_size}\0".encode()) if kind == "git" else hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 23), b""):
            h.update(chunk)
    return h.hexdigest()

def locate_model(spec, config_dirs):
    problems = []
    for d in config_dirs:
        try:
            if json.loads((d / "config.json").read_text()).get("model_type") != spec["model_type"]:
                continue
        except (OSError, ValueError):
            continue
        missing = [f for f in spec["files"] if not (d / f).is_file()]
        wrong = [f for f, (size, kind, digest) in spec["files"].items()
                 if f not in missing and ((d / f).stat().st_size != size or file_hash(d / f, kind) != digest)]
        if not missing and not wrong:
            return d
        problems.append(f"  {d}: missing {missing}, differs from official release {wrong}")
    raise FileNotFoundError(
        f"No verified local copy of {spec['name']} ({spec['official']}) under /kaggle/input.\n"
        f"Attach {spec['attach']} to the notebook (Add Input). Internet access is not used.\n"
        + ("\n".join(problems) if problems else "  (no directory with a matching config.json found)"))

INPUT = Path("/kaggle/input")
print("attached inputs:", sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else "none")
# os.walk(followlinks=True): Kaggle may mount inputs as symlinks, which Path.rglob does not descend into
config_dirs = sorted(Path(dp) for dp, _, fns in os.walk(INPUT, followlinks=True) if "config.json" in fns)
DEPTH_DIR = locate_model(MODEL_SPECS["depth"], config_dirs)
CLIP_DIR = locate_model(MODEL_SPECS["clip"], config_dirs)
print("Depth Anything V2 Small:", DEPTH_DIR)
print("CLIP ViT-B/32          :", CLIP_DIR)

depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_DIR, local_files_only=True).to(DEVICE).eval()
depth_proc = AutoImageProcessor.from_pretrained(DEPTH_DIR, local_files_only=True)
clip = CLIPModel.from_pretrained(CLIP_DIR, local_files_only=True).to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CLIP_DIR, local_files_only=True)

ids = proc.tokenizer("a photo of a human face")["input_ids"]
if (ids[0], ids[-1]) != (49406, 49407):
    raise RuntimeError(f"CLIP tokenizer loaded from {CLIP_DIR} does not match the original vocabulary: {ids}")
for name, model in (("Depth Anything", depth_model), ("CLIP", clip)):
    if next(model.parameters()).device.type != DEVICE:
        raise RuntimeError(f"{name} is not on {DEVICE}")
print(f"both models loaded on {DEVICE.upper()} | transformers {transformers.__version__} | "
      f"{type(depth_proc).__name__}, {type(proc.tokenizer).__name__}, {type(proc.image_processor).__name__}")

MODEL_INFO = {k: {"model": s["name"], "official_release": s["official"], "kaggle_source": s["attach"],
                  "local_dir": str(d), "verified_files": {f: f"{v[1]}:{v[2]}" for f, v in s["files"].items()}}
              for (k, s), d in zip(MODEL_SPECS.items(), (DEPTH_DIR, CLIP_DIR))}

LABELS = ["a photo of a human face", "a black and white image of a human face", "a photo of an animal",
          "a photo of an object", "a photo of a landscape", "an abstract black and white pattern",
          "a map", "an ink blot"]
FACE_IDX = [0, 1]
t_in = proc(text=LABELS, return_tensors="pt", padding=True)

def predict_depth(path):
    """Relative inverse depth at the input resolution (bicubic upsampling, as the HF depth pipeline does)."""
    im = Image.open(path).convert("RGB")
    x = depth_proc(images=im, return_tensors="pt")["pixel_values"].to(DEVICE)
    with torch.no_grad():
        pred = depth_model(pixel_values=x).predicted_depth
    d = torch.nn.functional.interpolate(pred.unsqueeze(1), size=(im.height, im.width), mode="bicubic", align_corners=False)
    return d.squeeze().float().cpu().numpy()

def clip_run(paths):
    embs, probs = [], []
    for i in range(0, len(paths), 32):
        x = proc(images=[Image.open(p).convert("RGB") for p in paths[i:i + 32]], return_tensors="pt")
        with torch.no_grad():
            out = clip(input_ids=t_in["input_ids"].to(DEVICE), attention_mask=t_in["attention_mask"].to(DEVICE),
                       pixel_values=x["pixel_values"].to(DEVICE))
        embs.append(out.image_embeds.cpu()); probs.append(out.logits_per_image.softmax(-1).cpu())
    return torch.cat(embs).numpy(), torch.cat(probs).numpy()

## Step 1 - select 100 side-lit faces
Mooney images work best with strong directional light (cast shadows break up the face).
Score = left/right brightness asymmetry of the face region + 0.5 * contrast; take the top 100 identities
whose Mooney keeps its white pixels on the face: faces where more than 1/3 of the white pixels fall outside
the central 60% x 60% region (i.e. on a bright background) are skipped.

In [ ]:
def find_lfw():
    jpgs = list(Path("/kaggle/input").rglob("*.jpg")) if Path("/kaggle/input").exists() else []
    if len(jpgs) < 1000:
        raise FileNotFoundError("LFW images not found under /kaggle/input - attach the Kaggle dataset "
                                "'jessicali9530/lfw-dataset' (internet access is not used).")
    deep = [p for p in jpgs if "deepfunneled" in str(p).lower()]
    return sorted(deep or jpgs)

def prep(path):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    c = int(min(h, w) * CROP)
    y0, x0 = (h - c) // 2, (w - c) // 2
    return cv2.resize(img[y0:y0 + c, x0:x0 + c], (SIZE, SIZE), interpolation=cv2.INTER_AREA)

def side_light_score(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    h, w = g.shape
    core = g[h // 5: 4 * h // 5, w // 5: 4 * w // 5]
    half = core.shape[1] // 2
    left, right = core[:, :half].mean(), core[:, half:].mean()
    return abs(left - right) / (left + right + 1e-6) + 0.5 * core.std()

def face_white_frac(img):
    m, _ = make_mooney(img, WHITE_PCT, BLUR_REL, SIGMA)
    h, w = m.shape
    white = m > 0
    return white[h // 5: 4 * h // 5, w // 5: 4 * w // 5].sum() / max(1, white.sum())

first_per_id = {}
for p in find_lfw():
    first_per_id.setdefault(p.parent.name, p)
print("identities:", len(first_per_id))

scored = sorted(((side_light_score(prep(p)), name, p) for name, p in first_per_id.items()), reverse=True)
rows, rejected = [], []
for score, name, p in scored:
    img = prep(p)
    fw = face_white_frac(img)
    if fw < MIN_FACE_WHITE:
        rejected.append({"identity": name, "side_light_score": round(score, 4), "face_white_frac": round(fw, 3)})
        continue
    stem = f"face_{len(rows) + 1:03d}"
    cv2.imwrite(str(OUT / "Original" / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    rows.append({"id": len(rows) + 1, "stem": stem, "identity": name, "source": p.name,
                 "side_light_score": round(score, 4), "face_white_frac": round(fw, 3)})
    if len(rows) == N_IMAGES:
        break
if len(rows) < N_IMAGES:
    raise RuntimeError(f"only {len(rows)} faces pass MIN_FACE_WHITE={MIN_FACE_WHITE:.2f}; lower it")
pd.DataFrame(rows).to_csv(ANA / "image_index.csv", index=False)
pd.DataFrame(rejected).to_csv(ANA / "rejected_bright_background.csv", index=False)
print(f"saved {len(rows)} originals | skipped {len(rejected)} higher-ranked faces with bright backgrounds")

## Step 2a - parameter sweep (white % x blur) to choose the Mooney settings

In [ ]:
orig_files = sorted((OUT / "Original").glob("*.jpg"))
whites, blurs = [16, 30, 45], [0.02, 0.04, 0.07]
samples = orig_files[:: max(1, len(orig_files) // 4)][:4]
fig, axes = plt.subplots(len(samples), 1 + len(whites) * len(blurs), figsize=(22, 2.6 * len(samples)))
for r, f in enumerate(samples):
    img = cv2.imread(str(f))
    axes[r, 0].imshow(img[..., ::-1]); axes[r, 0].set_title(f.stem, fontsize=8)
    c = 1
    for w in whites:
        for b in blurs:
            m, info = make_mooney(img, w, b, SIGMA)
            axes[r, c].imshow(m, cmap="gray", vmin=0, vmax=255)
            axes[r, c].set_title(f"white {w}% | k={info['ksize']}", fontsize=8)
            c += 1
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "sweep.png", dpi=110); plt.show()

## Step 2b - batch generation with the CLI tool

In [ ]:
!python mooney_tool.py {OUT/"Original"} {OUT/"Mooney"} --white {WHITE_PCT} --blur-rel {BLUR_REL} --sigma {SIGMA} --log {ANA/"mooney_params.csv"} | tee {ANA/"mooney_cli_log.txt"}

## Step 2c - interactive tool (sliders work in an interactive Kaggle session; use it for the tool screenshot)

In [ ]:
import ipywidgets as W

def show(idx=1, white=WHITE_PCT, blur_rel=BLUR_REL, invert=False):
    img = cv2.imread(str(orig_files[idx - 1]))
    m, info = make_mooney(img, white, blur_rel, SIGMA, invert=invert)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(img[..., ::-1]); ax[0].set_title(orig_files[idx - 1].stem)
    ax[1].imshow(m, cmap="gray", vmin=0, vmax=255)
    ax[1].set_title(f"k={info['ksize']}  white={info['white_pct_actual']}%")
    for a in ax:
        a.axis("off")
    plt.show()

W.interact(show,
           idx=W.IntSlider(value=1, min=1, max=len(orig_files), description="image"),
           white=W.FloatSlider(value=WHITE_PCT, min=5, max=70, step=1, description="white %"),
           blur_rel=W.FloatSlider(value=BLUR_REL, min=0.0, max=0.12, step=0.005, readout_format=".3f", description="blur"),
           invert=W.Checkbox(value=False, description="invert"));

## Contact sheets - use these to pick the top 10 (fill SUBSET above)

In [ ]:
def contact_sheet(files, path, cols=10, tile=128):
    rows_n = int(np.ceil(len(files) / cols))
    sheet = np.full((rows_n * tile, cols * tile, 3), 255, np.uint8)
    for k, f in enumerate(files):
        im = cv2.resize(cv2.imread(str(f)), (tile, tile), interpolation=cv2.INTER_AREA)
        cv2.putText(im, str(k + 1), (4, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
        r, c = divmod(k, cols)
        sheet[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile] = im
    cv2.imwrite(str(path), sheet)
    return sheet

mooney_files = sorted((OUT / "Mooney").glob("*.png"))
for name, files in (("sheet_original.jpg", orig_files), ("sheet_mooney.jpg", mooney_files)):
    s = contact_sheet(files, FIG / name)
    plt.figure(figsize=(14, 14)); plt.imshow(s[..., ::-1]); plt.axis("off"); plt.title(name); plt.show()

## Pre-flight check - both models on one original and one Mooney image before the full runs

In [ ]:
watchdog(10, "pre-flight inference check")
for f in (orig_files[0], mooney_files[0]):
    t = time.time()
    d = predict_depth(f)
    if d.shape != (SIZE, SIZE) or not np.isfinite(d).all() or d.std() == 0:
        raise RuntimeError(f"Depth Anything produced an invalid depth map for {f.name}: shape {d.shape}")
    t_depth = time.time() - t
    e, p = clip_run([f])
    if e.shape != (1, 512) or not np.isfinite(e).all() or abs(float(p.sum()) - 1) > 1e-3:
        raise RuntimeError(f"CLIP produced invalid output for {f.name}: embedding shape {e.shape}")
    print(f"OK {f.parent.name}/{f.name} on {DEVICE}: depth {d.shape} range [{d.min():.2f}, {d.max():.2f}] "
          f"({t_depth:.1f}s) | CLIP top-1 '{LABELS[int(p.argmax())]}' ({p.max():.2f})", flush=True)

## Step 3 - Vision system 1: Depth Anything V2 Small
Output is relative inverse depth (bright = near). Saved per image as 16-bit PNG + colour visualisation.
Metric: pixel-wise correlation between the depth of the original and of its Mooney version,
compared with a control of mismatched pairs (original i vs. Mooney i+1).

In [ ]:
watchdog(60, "Depth Anything on 200 images")
depth = {}
for kind, files in (("Original", orig_files), ("Mooney", mooney_files)):
    dst = OUT / "Result1" / kind
    dst.mkdir(parents=True, exist_ok=True)
    depth[kind] = {}
    for j, f in enumerate(files, 1):
        d = predict_depth(f)
        n = (d - d.min()) / (d.max() - d.min() + 1e-8)
        cv2.imwrite(str(dst / f"{f.stem}_depth16.png"), (n * 65535).astype(np.uint16))
        cv2.imwrite(str(dst / f"{f.stem}_depth_vis.jpg"), cv2.applyColorMap((n * 255).astype(np.uint8), cv2.COLORMAP_INFERNO))
        depth[kind][f.stem] = d
        if j % 25 == 0:
            print(f"[{time.time() - T0:7.1f}s] depth {kind} {j}/{len(files)}", flush=True)
    print(kind, "done:", len(files))

stems = sorted(depth["Original"])
dm = []
for i, s in enumerate(stems):
    a, b = depth["Original"][s].ravel(), depth["Mooney"][s].ravel()
    c = depth["Mooney"][stems[(i + 1) % len(stems)]].ravel()
    dm.append({"stem": s, "pearson": pearsonr(a, b)[0], "spearman": spearmanr(a[::7], b[::7])[0],
               "pearson_mismatched_control": pearsonr(a, c)[0]})
dm = pd.DataFrame(dm)
dm.to_csv(ANA / "depth_metrics.csv", index=False)
print(dm.describe().round(3))

## Step 4 - Vision system 2: CLIP ViT-B/32
(a) zero-shot labelling - does CLIP still see a face in the Mooney image?
(b) 8AFC identity matching exactly like the paper: probe vs. 8 gallery images, cosine similarity, chance = 12.5%.

In [ ]:
watchdog(20, "CLIP on 200 images")
clip_emb, clip_rows = {}, []
for kind, files in (("Original", orig_files), ("Mooney", mooney_files)):
    dst = OUT / "Result2" / kind
    dst.mkdir(parents=True, exist_ok=True)
    emb, prob = clip_run(files)
    clip_emb[kind] = emb
    for f, p in zip(files, prob):
        res = {"image": f.name, "top1": LABELS[int(p.argmax())], "p_face": float(p[FACE_IDX].sum()),
               "probs": {l: round(float(v), 4) for l, v in zip(LABELS, p)}}
        (dst / f"{f.stem}.json").write_text(json.dumps(res, indent=2))
        clip_rows.append({"kind": kind, "stem": f.stem, "top1": res["top1"], "p_face": res["p_face"]})
np.savez(ANA / "clip_embeddings.npz", **clip_emb)
cr = pd.DataFrame(clip_rows)
cr.to_csv(ANA / "clip_zero_shot.csv", index=False)
print(cr.groupby("kind")["p_face"].describe().round(3))
print(cr.groupby("kind")["top1"].value_counts())

In [ ]:
def afc8(probe, gallery, seed=SEED):
    rng = np.random.default_rng(seed)
    n, correct = len(probe), 0
    for i in range(n):
        g = np.concatenate([[i], rng.choice(np.delete(np.arange(n), i), 7, replace=False)])
        correct += int(np.argmax(gallery[g] @ probe[i]) == 0)
    return correct, n

def unit(x):
    x = x - x.mean(1, keepdims=True)
    return x / np.linalg.norm(x, axis=1, keepdims=True)

dvec = {k: unit(np.stack([cv2.resize(depth[k][s], (64, 64)).ravel() for s in stems])) for k in depth}
afc_rows = []
for system, emb in (("CLIP embedding", clip_emb), ("Depth Anything depth map", dvec)):
    for probe, gallery in (("Original", "Mooney"), ("Mooney", "Original")):
        k, n = afc8(emb[probe], emb[gallery])
        afc_rows.append({"system": system, "probe": probe, "gallery": gallery, "correct": k, "n": n,
                         "accuracy": k / n, "p_vs_chance": binomtest(k, n, 0.125, alternative="greater").pvalue})
afc = pd.DataFrame(afc_rows)
afc.to_csv(ANA / "afc8_results.csv", index=False)
print(afc.round(4).to_string(index=False))
watchdog(15, "validation, subset, packaging")

## Validation - all 200 images processed, Mooney constraint holds; writes analysis/run_info.json

In [ ]:
import platform
from datetime import datetime, timezone

checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": str(detail)})
    print("PASS" if ok else "FAIL", name, "|", detail)

def stems_with(folder, suffix):
    return sorted(p.name[: -len(suffix)] for p in folder.glob(f"*{suffix}"))

stems_o, stems_m = stems_with(OUT / "Original", ".jpg"), stems_with(OUT / "Mooney", ".png")
check("100 original images", len(stems_o) == N_IMAGES, len(stems_o))
check("100 Mooney images, same names as originals", stems_m == stems_o, len(stems_m))
masks = [cv2.imread(str(OUT / "Mooney" / f"{s}.png"), cv2.IMREAD_GRAYSCALE) for s in stems_m]
white = np.array([100.0 * (m == 255).mean() for m in masks])
check("Mooney images strictly black/white", all(set(np.unique(m)) <= {0, 255} for m in masks))
check(f"white pixels = {WHITE_PCT:.1f}% in every Mooney image", np.abs(white - WHITE_PCT).max() < 0.05,
      f"min {white.min():.3f}%, max {white.max():.3f}%")
params = pd.read_csv(ANA / "mooney_params.csv")
check("Mooney parameters logged for every image", len(params) == N_IMAGES,
      f"images needing tie-fill: {params.loc[params['tie_filled_px'] > 0, 'file'].tolist()}")
for kind in ("Original", "Mooney"):
    r1, r2 = OUT / "Result1" / kind, OUT / "Result2" / kind
    check(f"Result1/{kind}: depth map + visualisation for every image",
          stems_with(r1, "_depth16.png") == stems_o and stems_with(r1, "_depth_vis.jpg") == stems_o,
          f"{len(stems_with(r1, '_depth16.png'))} + {len(stems_with(r1, '_depth_vis.jpg'))} files")
    check(f"Result2/{kind}: CLIP result for every image", stems_with(r2, ".json") == stems_o,
          f"{len(stems_with(r2, '.json'))} files")
check("depth metrics for all 100 pairs", len(dm) == N_IMAGES and dm["pearson"].notna().all(), len(dm))
check("CLIP zero-shot results for all 200 images", len(cr) == 2 * N_IMAGES, len(cr))
check("8AFC results (2 systems x 2 directions)", len(afc) == 4, len(afc))

run_info = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "device": DEVICE,
    "offline_env": {k: os.environ.get(k) for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE")},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "opencv": cv2.__version__, "numpy": np.__version__},
    "data": {"source": "Kaggle dataset jessicali9530/lfw-dataset (LFW deep-funneled)", "n_images": N_IMAGES,
             "size": SIZE, "crop": CROP, "min_face_white": MIN_FACE_WHITE},
    "mooney": {"white_pct": WHITE_PCT, "blur_rel": BLUR_REL, "sigma": SIGMA},
    "models": MODEL_INFO,
    "checks": checks,
}
(ANA / "run_info.json").write_text(json.dumps(run_info, indent=2))
failed = [c["check"] for c in checks if not c["passed"]]
if failed:
    raise RuntimeError(f"Validation failed: {failed}")
print(f"all {len(checks)} checks passed")

## Subset (top 10) + slide figures - only after SUBSET has been filled

In [ ]:
if len(SUBSET) == 10:
    sub = [f"face_{i:03d}" for i in SUBSET]
    for kind, ext in (("Original", ".jpg"), ("Mooney", ".png")):
        d = OUT / "Subset" / kind
        d.mkdir(parents=True, exist_ok=True)
        for s in sub:
            shutil.copy(OUT / kind / f"{s}{ext}", d / f"{s}{ext}")
    fig, ax = plt.subplots(2, 10, figsize=(20, 4.4))
    for j, s in enumerate(sub):
        ax[0, j].imshow(cv2.imread(str(OUT / "Original" / f"{s}.jpg"))[..., ::-1]); ax[0, j].set_title(s, fontsize=9)
        ax[1, j].imshow(cv2.imread(str(OUT / "Mooney" / f"{s}.png"), 0), cmap="gray")
    for a in ax.ravel():
        a.axis("off")
    plt.tight_layout(); plt.savefig(FIG / "subset_images.png", dpi=150); plt.show()

    pf = cr.pivot(index="stem", columns="kind", values="p_face")
    for part, chunk in enumerate((sub[:5], sub[5:]), 1):
        fig, ax = plt.subplots(5, 4, figsize=(10, 13))
        for r, s in enumerate(chunk):
            r_depth = dm.set_index("stem").loc[s, "pearson"]
            panels = [(cv2.imread(str(OUT / "Original" / f"{s}.jpg"))[..., ::-1], f"{s} original"),
                      (cv2.imread(str(OUT / "Mooney" / f"{s}.png"))[..., ::-1], "Mooney"),
                      (cv2.imread(str(OUT / "Result1/Original" / f"{s}_depth_vis.jpg"))[..., ::-1],
                       f"depth orig | CLIP p(face)={pf.loc[s, 'Original']:.2f}"),
                      (cv2.imread(str(OUT / "Result1/Mooney" / f"{s}_depth_vis.jpg"))[..., ::-1],
                       f"depth Mooney r={r_depth:.2f} | p(face)={pf.loc[s, 'Mooney']:.2f}")]
            for c, (im, title) in enumerate(panels):
                ax[r, c].imshow(im); ax[r, c].set_title(title, fontsize=8); ax[r, c].axis("off")
        plt.tight_layout(); plt.savefig(FIG / f"subset_results_{part}.png", dpi=130); plt.show()
else:
    print("SUBSET not set yet - pick 10 ids from figures/sheet_mooney.jpg, fill SUBSET and re-run.")

## Packaging - one ZIP with all images, results, analysis, metadata and the tool code; verified after writing

In [ ]:
import zipfile
(EXPORT / "code").mkdir(exist_ok=True)
shutil.copy("mooney_tool.py", EXPORT / "code" / "mooney_tool.py")
zip_path = Path(shutil.make_archive(str(WORK / "part1_output"), "zip", EXPORT)).resolve()

with zipfile.ZipFile(zip_path) as z:
    names = [n.removeprefix("./") for n in z.namelist() if not n.endswith("/")]
    corrupt = z.testzip()
count = lambda prefix, suffix: sum(n.startswith(prefix) and n.endswith(suffix) for n in names)
expected = {("deliverable/Original/", ".jpg"): N_IMAGES, ("deliverable/Mooney/", ".png"): N_IMAGES}
for kind in ("Original", "Mooney"):
    expected[(f"deliverable/Result1/{kind}/", "_depth16.png")] = N_IMAGES
    expected[(f"deliverable/Result1/{kind}/", "_depth_vis.jpg")] = N_IMAGES
    expected[(f"deliverable/Result2/{kind}/", ".json")] = N_IMAGES
required = ["analysis/run_info.json", "analysis/image_index.csv", "analysis/rejected_bright_background.csv",
            "analysis/mooney_params.csv",
            "analysis/mooney_cli_log.txt", "analysis/depth_metrics.csv", "analysis/clip_zero_shot.csv",
            "analysis/clip_embeddings.npz", "analysis/afc8_results.csv", "figures/sweep.png",
            "figures/sheet_original.jpg", "figures/sheet_mooney.jpg", "code/mooney_tool.py"]
problems = [f"{p}*{s}: {count(p, s)} files, expected {n}" for (p, s), n in expected.items() if count(p, s) != n]
problems += [f"missing {r}" for r in required if r not in names]
if corrupt:
    problems.append(f"corrupt member {corrupt}")
if zip_path.stat().st_size == 0:
    problems.append("ZIP file is empty")
print(f"ZIP: {zip_path} | {zip_path.stat().st_size / 1e6:.1f} MB | {len(names)} files")
if problems:
    raise RuntimeError("ZIP verification failed:\n" + "\n".join(problems))
for (p, s), n in expected.items():
    print(f"  {p}*{s}: {count(p, s)}")
print("ZIP verified: all expected outputs present, CRC check passed")
faulthandler.cancel_dump_traceback_later()
print(f"total runtime {time.time() - T0:.0f}s")